# M5 — App y despliegue (Turno Juan Esteban) — validación en Kaggle
### SI4006 · Proyecto Integrador · Universidad EAFIT

Este notebook levanta `m5/app/app.py` (Gradio) sobre el sistema completo de M1+M2+M3+M4
en una GPU T4 de Kaggle, con link público (`share=True`) para la demo en vivo del
jueves 5 nov.

**Antes de correr, en Kaggle:**
- *Settings → Accelerator* → GPU T4 x2 (o x1, alcanza).
- *Settings → Internet* → On (necesita cuenta verificada por teléfono).
- Este notebook clona la rama `juanes-m5` del repo. **Debe estar pusheada a GitHub**
  antes de correr esto (`git push origin juanes-m5` desde el local), si no, el clone
  falla.


## 1 · Preparar el entorno

In [ ]:
import os

RAMA = "juanes-m5"

if os.path.isdir("triage-dermatologico-ia"):
    %cd triage-dermatologico-ia
    !git checkout {RAMA}
    !git pull origin {RAMA}
else:
    !git clone --branch {RAMA} https://github.com/Isa-Idarraga/triage-dermatologico-ia.git
    %cd triage-dermatologico-ia

!git log -1 --oneline  # confirmar a simple vista qué commit quedó activo


In [ ]:
%pip install -q -r requirements.txt

# Mismo arreglo que en los notebooks de M2/M3/M4: el entorno trae una version vieja
# de torchao preinstalada que hace tronar a `peft` al importarse.
!pip uninstall -y -q torchao


In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: no hay GPU activa. Ve a Settings -> Accelerator -> GPU T4, "
          "y vuelve a correr esta celda.")


## 2 · Cargar la app y calentar los modelos

`_calentar()` fuerza la carga de los 4 modelos (cerebro, BETO, RAG, `ver()`) una sola
vez, antes de que el primer enfermero haga una consulta real. `health()` todavía no
tiene nada real que reportar -- `m5/observabilidad.py` es de María Alejandra y lo
entrega el lunes 2 nov. Mientras tanto, `app.py` usa un *fallback* que no rompe nada;
en el momento en que ella suba su archivo, esto queda enchufado sin tocar código.

In [ ]:
import sys
sys.path.insert(0, "m5/app")
import app

print("Versión (tag de Git):", app._version_tag())
print("health():", app.health())

app._calentar()


## 3 · Probar el backend (`consultar`) antes de levantar la interfaz

Dos casos ya conocidos de M4, para confirmar que la tarjeta arma bien las tres
señales reales (no los mocks que usamos en local sin GPU):
- **Sin foto**: un síntoma que BETO y el RAG deberían marcar urgente.
- **Con foto**: la imagen `ISIC_0024876.jpg` (nevus benigno confirmado por biopsia) +
  un síntoma vago -- el caso real donde `ver_imagen` y el RAG acertaron en
  'no_urgente' pero BETO forzó 'urgente' y la regla de seguridad escaló el veredicto.
  Debe verse exactamente esa lectura en el bloque de decisión.

In [ ]:
decision, bloque_foto, evidencia = app.consultar(
    "Tengo un lunar que me cambio de color y de forma en los ultimos meses.", None,
)
print(decision)
print()
print(bloque_foto)

assert "URGENTE" in decision, f"se esperaba URGENTE (BETO y RAG deberian coincidir), salio: {decision}"
assert "No evaluable: no se uso" in bloque_foto, "sin foto, el bloque de foto debe decir que no se uso"
print()
print("OK: coincide con la corrida de referencia en Kaggle T4.")


In [ ]:
_imagen_prueba = "m4/pipeline/eval_tool/imagenes/ISIC_0024876.jpg"
assert os.path.exists(_imagen_prueba), "no se encontro la imagen de prueba del eval_tool"

decision, bloque_foto, evidencia = app.consultar(
    "Le adjunto una foto de la lesion, deberia verla pronto?", _imagen_prueba,
)
print(decision)
print()
print(bloque_foto)
print()
print(evidencia)

# Valores de referencia: misma imagen, mismo resultado ya confirmado en Colab (M4) y
# en esta corrida de Kaggle -- ver_imagen() es deterministico (sin aleatoriedad) y el
# agente usa greedy decoding (do_sample=False), asi que debe repetirse exacto.
assert "URGENTE" in decision, f"la regla de seguridad deberia escalar a URGENTE, salio: {decision}"
assert "ver_imagen=no_urgente" in decision, f"ver_imagen deberia decir no_urgente en esta imagen: {decision}"
assert "clasificador_beto=urgente" in decision, f"BETO deberia forzar urgente con un sintoma vago: {decision}"
assert "0.157" in bloque_foto, f"el score de ver() deberia ser ~0.1565 (0.157 redondeado): {bloque_foto}"
assert "no describe su foto" in evidencia.lower(), "falta la nota de que la evidencia del RAG no describe la foto"
print()
print("OK: coincide con la corrida de referencia en Kaggle T4 y con el hallazgo de M4.")


### Caso borde: foto no evaluable (no debe votar "no urgente")

`img_06.jpg` del eval set de M4 está desenfocada a propósito (ya lo confirmó Camilo sin
GPU, con el filtro de calidad puro de `ver()`). El contrato es: si la foto no se puede
evaluar, `etiqueta` queda en `None` y **no cuenta como evidencia de "no_urgente"** --
solo aparece el aviso. Esto se verifica con un `assert`, no mirando la UI a ojo.

In [ ]:
_imagen_borrosa = "m4/eval_set/imagenes/img_06.jpg"
assert os.path.exists(_imagen_borrosa), "no se encontro img_06 (foto desenfocada) del eval set"

decision_borrosa, bloque_foto_borrosa, _ = app.consultar(
    "Tengo una mancha rara en la espalda, no se si es grave.", _imagen_borrosa,
)
print(decision_borrosa)
print()
print(bloque_foto_borrosa)

assert "No evaluable" in bloque_foto_borrosa, "la foto borrosa deberia marcar 'No evaluable', no dar una clase"
assert "no equivale" in bloque_foto_borrosa.lower(), "falta el aviso de que no equivale a 'no urgente'"
print()
print("OK: la foto no evaluable no se conto como evidencia de 'no_urgente'.")


## 4 · Levantar la interfaz con link público

Los tres casos (sin foto, con foto benigna, foto no evaluable) ya quedaron probados y
verificados con `assert` en las celdas de arriba -- con "Run All" alcanza, no hay que
mirarlos a mano. Esta celda final solo levanta la UI para que puedas *verla* y usarla
en vivo (y es la que se deja corriendo el jueves de la demo).

`prevent_thread_lock=True` hace que la app quede corriendo en segundo plano sin
bloquear la celda, así "Run All" sí termina. El link público queda impreso abajo;
la sesión de Kaggle debe seguir activa para que el link siga vivo.

**Cuando termines de probar y no vayas a seguir usando el link, para la sesión de
Kaggle** (o corre `demo.close()` en una celda nueva) para no seguir gastando las
~30 h semanales de GPU.

In [ ]:
demo = app.construir_app()
demo.launch(share=True, prevent_thread_lock=True)
print()
print("App corriendo en segundo plano -- usa el link https://....gradio.live de arriba.")


---
Con esto queda probada la app de punta a punta en una T4 real de Kaggle: carga única
de los 4 modelos, la tarjeta de 3 bloques con casos reales (no mocks), y el link
público que se usará el jueves. Pendiente: enchufar `health()`/`log_consulta()` reales
en cuanto María Alejandra suba `m5/observabilidad.py`, y repetir esta corrida si
Camilo confirma que 4 bits gana (cambiaría qué modelo carga `_calentar()`).